# Análise de vídeo cirúrgico com YOLOv8 — CholecT50 (Google Colab)

Tech Challenge Fase 4 · FIAP IA para Devs · requisito **Análise de Vídeo**: processar vídeo clínico,
detectar eventos fora do padrão com YOLOv8 e gerar relatório automático.

**O que este notebook faz, sem você enviar nenhum vídeo**
1. Baixa o pacote oficial **CholecT50 Challenge Validation** (repositório do CAMMA) direto no Colab.
2. Converte os rótulos (caixas dos instrumentos) para o formato YOLO, dividindo **por clipe**.
3. Ajusta o **YOLOv8** para detectar instrumentos cirúrgicos e avalia em um clipe **nunca visto**.
4. Monta um vídeo de demonstração com o clipe de teste, detecta os instrumentos e calcula indicadores
   fora do padrão (instrumento fora do campo, movimento brusco, ausência prolongada).
5. Gera o relatório HTML e (opcional) integra com o alerta multimodal.

**Antes de rodar:** *Ambiente de execução → Alterar tipo → GPU (T4)*. Execute as células em ordem.

> ⚠️ **Dataset:** licença CC BY-NC-SA 4.0 (uso não comercial, citar Nwoye et al., *Rendezvous*, Medical Image
> Analysis 2022). A página do desafio CholecTriplet2022 informa que a **publicação de resultados sobre este
> conjunto de validação não era permitida até a publicação conjunta do desafio**; confira a situação atual
> antes de divulgar o vídeo/relatório fora da disciplina. As caixas deste pacote vêm do dataset
> m2cai16-tool-location.
>
> ⚠️ **Escala:** são **5 clipes curtos (~1,1 mil frames)**. Serve para demonstrar o fluxo e uma avaliação inicial,
> mas as métricas são pouco robustas e variam muito com o clipe escolhido para teste.

## 1. Código do projeto e dependências

O notebook **clona o repositório do GitHub** sozinho: só é preciso preencher `REPO_URL` na célula abaixo.
Se o repositório for **privado**, crie no Colab (ícone 🔑 *Segredos*) o segredo `GITHUB_TOKEN` com um token de
leitura (fine-grained, permissão *Contents: read*) e ative *Acesso ao notebook*. Ao reexecutar, o código é atualizado
com `git pull`.


In [ ]:
import os, sys, shutil, subprocess, zipfile, pathlib, json, time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# --- Código do projeto: o notebook clona o repositório do GitHub (não é preciso enviar nenhum arquivo) ---
REPO_URL = "https://github.com/SEU-USUARIO/multimodal-health-monitor.git"   # <- troque pelo URL do seu repositório
REPO_BRANCH = None                  # None = branch padrão do repositório (ex.: "main")
GITHUB_TOKEN_SECRET = "GITHUB_TOKEN"  # só para repositório PRIVADO: segredo do Colab (ícone 🔑) com um token de leitura

REPO_NAME = REPO_URL.rstrip("/").removesuffix(".git").split("/")[-1]


def _git(*args, cwd=None, mask=None):
    r = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    if r.returncode:
        msg = (r.stderr or r.stdout).strip()
        raise RuntimeError(f"git {args[0]} falhou: " + (msg.replace(mask, "***") if mask else msg))
    return r.stdout.strip()


def _github_token():
    try:
        from google.colab import userdata
        return userdata.get(GITHUB_TOKEN_SECRET) or None
    except Exception:           # segredo inexistente ou sem acesso ao notebook
        return None


if IN_COLAB:
    PROJECT_DIR = f"/content/{REPO_NAME}"
    if "SEU-USUARIO" in REPO_URL:
        raise ValueError("Defina REPO_URL com o URL do seu repositório no GitHub (célula acima).")
    token = _github_token()
    url = REPO_URL.replace("https://", f"https://x-access-token:{token}@", 1) if token else REPO_URL
    branch_args = ["--branch", REPO_BRANCH] if REPO_BRANCH else []
    if not os.path.isdir(os.path.join(PROJECT_DIR, ".git")):
        try:
            _git("clone", "--depth", "1", *branch_args, url, PROJECT_DIR, mask=token)
        except RuntimeError as e:
            raise RuntimeError(f"{e}\nSe o repositório for PRIVADO, crie o segredo {GITHUB_TOKEN_SECRET} no Colab "
                               "(ícone 🔑), com um token de leitura, e ative 'Acesso ao notebook'.") from None
        if token:                                      # não deixa o token gravado no .git/config
            _git("remote", "set-url", "origin", REPO_URL, cwd=PROJECT_DIR)
    else:                                              # sessão já tem o código: traz as atualizações
        try:
            _git("pull", "--ff-only", url, *([REPO_BRANCH] if REPO_BRANCH else ["HEAD"]), cwd=PROJECT_DIR, mask=token)
        except RuntimeError as e:
            print("Aviso: não consegui atualizar o código; usando a versão que já está na sessão.\n", e)
else:   # local: raiz do repositório (ou a pasta pai, se o Jupyter abriu em notebooks/)
    PROJECT_DIR = os.getcwd() if os.path.isdir("src") else os.path.dirname(os.getcwd())

os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
assert os.path.isdir("src"), f"Pasta src/ não encontrada em {os.getcwd()} — confira REPO_URL."
try:
    print("Projeto em:", os.getcwd(), "| Colab:", IN_COLAB, "| commit:", _git("rev-parse", "--short", "HEAD"))
except Exception:
    print("Projeto em:", os.getcwd(), "| Colab:", IN_COLAB)

In [ ]:
# O Colab já tem torch, opencv, pandas e matplotlib; instalamos o restante.
if IN_COLAB:
    %pip install -q ultralytics pyyaml scipy python-dotenv
else:
    print("Fora do Colab: instale com `pip install -r requirements.txt`.")

## 2. Parâmetros

In [ ]:
# Cada clipe do CholecT50 é tratado como UM paciente distinto (ID = prefixo + nome do clipe).
# Não há identificação de paciente no pacote; ver docs/cholect50.md. Não reaproveitamos um ID fixo.
ID_PREFIX = "CHOLECT"
def patient_id_for(clip_name):
    return f"{ID_PREFIX}-{clip_name}"

# --- Dataset: pacote oficial com download direto e caixas de instrumento em todos os clipes ---
DATASET_ZIP_URL = "https://s3.unistra.fr/camma_public/datasets/cholect50/CholecT50-Challenge-Validation.zip"
LOCAL_ZIP = None                     # se o download falhar: caminho de um .zip já salvo (ex.: no Drive)
RAW_DIR = "data/cholect50_raw"       # onde extrair
YOLO_DIR = "data/cholect50_yolo"     # dataset convertido

# --- Divisão por clipe (None = automática: ordem alfabética; penúltimo = val, último = teste) ---
VAL_CLIPS = None                     # ex.: ["VID96"]
TEST_CLIPS = None                    # ex.: ["VID111"]

# --- Treino do YOLOv8 ---
BASE_MODEL = "yolov8n.pt"            # yolov8s.pt = mais preciso e mais lento
EPOCHS = 50
IMGSZ, BATCH, WORKERS = 640, 16, 2   # se faltar memória na GPU: IMGSZ=480 ou BATCH=8
PLOTS = True                         # curvas e matriz de confusão do Ultralytics

# --- Análise do clipe de teste ---
CONF = 0.35                          # confiança mínima das detecções
MAX_IDLE_S = 15                      # alerta se ficar >= N FRAMES ADJACENTES sem instrumento (câmera dentro do corpo)
RENDER_VIDEO = True                  # gerar vídeo anotado para assistir no notebook
RUN_FUSION = True                    # integra o clipe de teste com sinais vitais SINTÉTICOS do mesmo ID e dispara o alerta (seção 7)

USE_DRIVE = False                    # True = salva resultados no Google Drive (seção 8)
DRIVE_DIR = "/content/drive/MyDrive/cholect50_resultados"

In [ ]:
import cv2, yaml, logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, Image, Video, display

from scripts.prepare_cholect50_yolo import (classify_frame, discover_sources, frames_to_video,
                                            inspect_dataset, prepare)

PALETTE = [(255, 99, 71), (30, 144, 255), (50, 205, 50), (255, 215, 0), (186, 85, 211), (0, 206, 209), (255, 140, 0)]


def findings_table(findings):
    rows = [f.to_dict() for f in findings]
    cols = ["modality", "kind", "score", "description"]
    return pd.DataFrame(rows)[cols].sort_values("score", ascending=False) if rows else pd.DataFrame(columns=cols)


def draw_boxes(rgb, boxes, names, thickness=2, color=None, prefix=""):
    """Desenha caixas YOLO normalizadas (classe, cx, cy, w, h) sobre uma imagem RGB."""
    h, w = rgb.shape[:2]
    for cid, cx, cy, bw, bh in boxes:
        x1, y1 = int((cx - bw / 2) * w), int((cy - bh / 2) * h)
        x2, y2 = int((cx + bw / 2) * w), int((cy + bh / 2) * h)
        col = color or PALETTE[cid % len(PALETTE)]
        cv2.rectangle(rgb, (x1, y1), (x2, y2), col, thickness)
        label = f"{prefix}{names[cid] if cid < len(names) else cid}"
        cv2.putText(rgb, label, (x1, max(12, y1 - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, col, 1, cv2.LINE_AA)
    return rgb


def read_rgb(path):
    return cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)


def read_yolo_labels(path):
    rows = [l.split() for l in Path(path).read_text().splitlines() if l.strip()]
    return [(int(r[0]), *map(float, r[1:5])) for r in rows]

## 3. Download do dataset oficial

Baixa o pacote **CholecT50 Challenge Validation** do servidor do CAMMA (link da página de *Downloads* do
repositório `CAMMA-public/cholect50`). O dataset completo (50 vídeos) só é liberado mediante formulário de
solicitação; este pacote é o subconjunto de download direto.

In [ ]:
import urllib.request


def download(url, dest, retries=3):
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (tech-challenge-colab)"})
    for attempt in range(1, retries + 1):
        try:
            with urllib.request.urlopen(req, timeout=60) as r, open(dest, "wb") as f:
                total = int(r.headers.get("Content-Length") or 0)
                done, last = 0, 0.0
                while True:
                    chunk = r.read(1 << 20)
                    if not chunk:
                        break
                    f.write(chunk)
                    done += len(chunk)
                    if time.time() - last > 5:
                        print(f"\r  {done / 1e6:8.1f} MB" + (f" / {total / 1e6:.1f} MB" if total else ""), end="")
                        last = time.time()
            print(f"\n  concluído: {done / 1e6:.1f} MB")
            return dest
        except Exception as e:  # noqa: BLE001
            print(f"\n  tentativa {attempt}/{retries} falhou: {e}")
            time.sleep(3 * attempt)
    raise RuntimeError(
        "Não consegui baixar o dataset. Baixe o ZIP manualmente pela página de Downloads do repositório "
        "CAMMA-public/cholect50, salve-o (ex.: no Drive) e informe o caminho em LOCAL_ZIP.")


raw, marker = Path(RAW_DIR), Path(RAW_DIR) / ".extracted"
if marker.exists():
    print("Dataset já extraído em", raw)
else:
    zip_path = Path(LOCAL_ZIP) if LOCAL_ZIP else download(DATASET_ZIP_URL, "data/downloads/cholect50_challenge_validation.zip")
    if not zipfile.is_zipfile(zip_path):
        raise RuntimeError(f"{zip_path} não é um ZIP válido (o servidor pode ter devolvido uma página de erro).")
    raw.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(raw)
    marker.write_text("ok")
print("Arquivos extraídos:", sum(1 for p in raw.rglob("*") if p.is_file() and not p.name.startswith(".")))

### 3.1 Estrutura baixada

O conversor descobre sozinho onde estão os rótulos e os frames. Esta saída mostra o que veio no pacote
(se algo falhar adiante, copie este bloco — ele diz exatamente o que ajustar).

In [ ]:
print(inspect_dataset(RAW_DIR))

## 4. Conversão para YOLO e exploração

Cada clipe vira treino, validação **ou** teste (nunca dividimos frames do mesmo clipe, pois frames vizinhos são
quase idênticos e inflariam as métricas). Frames pretos (anonimizados, fora do corpo) e frames com
instrumento sem caixa são descartados.

In [ ]:
stats = prepare(Path(RAW_DIR), Path(YOLO_DIR), val=VAL_CLIPS, test=TEST_CLIPS)

In [ ]:
cfg = yaml.safe_load(Path(YOLO_DIR, "cholect50_yolo.yaml").read_text())
NAMES = [cfg["names"][i] for i in sorted(cfg["names"])]

summary = pd.DataFrame({s: {k: d[k] for k in ("videos", "images", "positives", "negatives", "ambiguos_descartados")}
                        for s, d in stats["splits"].items()}).T
display(summary)

rows = [{"split": s, "classe": c, "caixas": n} for s, d in stats["splits"].items()
        for c, n in d["boxes_per_class"].items()]
dist = pd.DataFrame(rows).pivot(index="classe", columns="split", values="caixas").fillna(0)
dist.plot.bar(figsize=(9, 3.8), title="Caixas de instrumento por classe e split")
plt.ylabel("caixas"); plt.tight_layout(); plt.show()

In [ ]:
# Amostras de treino com as caixas anotadas
imgs = sorted(Path(YOLO_DIR, "images", "train").glob("*"))
rng = np.random.default_rng(0)
pick = rng.choice(len(imgs), size=min(6, len(imgs)), replace=False)
fig, axes = plt.subplots(2, 3, figsize=(15, 6))
for ax, i in zip(axes.ravel(), pick):
    p = imgs[i]
    ax.imshow(draw_boxes(read_rgb(p), read_yolo_labels(Path(YOLO_DIR, "labels", "train", p.stem + ".txt")), NAMES))
    ax.set_title(p.stem, fontsize=9); ax.axis("off")
plt.suptitle("Treino: frames com caixas anotadas"); plt.tight_layout(); plt.show()

## 5. Treino e avaliação do YOLOv8

Ajuste fino a partir de pesos pré-treinados (COCO). Flips e pequenas rotações são usados porque imagens
endoscópicas não têm "cima" fixo e o conjunto é pequeno. A avaliação final usa o clipe de **teste**.

In [ ]:
import torch
DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("Dispositivo de treino:", "GPU" if DEVICE == 0 else "CPU (lento — ative a GPU no Colab)")
PLOT_FLAG = "" if PLOTS else "--no-plots"

In [ ]:
!python scripts/train_yolo_cholect50.py --data {YOLO_DIR}/cholect50_yolo.yaml --model {BASE_MODEL} --epochs {EPOCHS} --imgsz {IMGSZ} --batch {BATCH} --workers {WORKERS} --device {DEVICE} {PLOT_FLAG}

### 5.1 Curvas de treino e métricas no clipe de teste

In [ ]:
runs = sorted(Path("runs").rglob("results.csv"), key=lambda p: p.stat().st_mtime)
if runs:
    res = pd.read_csv(runs[-1])
    res.columns = [c.strip() for c in res.columns]
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
    for k in ("train/box_loss", "val/box_loss"):
        if k in res: ax[0].plot(res["epoch"], res[k], label=k)
    for k in ("train/cls_loss", "val/cls_loss"):
        if k in res: ax[1].plot(res["epoch"], res[k], label=k)
    for k in ("metrics/mAP50(B)", "metrics/mAP50-95(B)"):
        if k in res: ax[2].plot(res["epoch"], res[k], label=k)
    for a, t in zip(ax, ("Perda de caixa", "Perda de classe", "mAP na validação")):
        a.set_title(t); a.set_xlabel("época"); a.legend(fontsize=8)
    plt.tight_layout(); plt.show()
    run_dir = runs[-1].parent
    for fname in ("confusion_matrix_normalized.png", "confusion_matrix.png"):
        if (run_dir / fname).exists():
            display(Image(filename=str(run_dir / fname), width=560)); break
else:
    print("Nenhum results.csv encontrado — o treino rodou?")

metrics = Path("models/test_metrics.json")
if metrics.exists():
    m = json.loads(metrics.read_text())
    display(pd.Series({k: v for k, v in m.items() if k != "AP50_por_classe"}).to_frame("clipe de TESTE (nunca visto)"))
    display(pd.Series(m["AP50_por_classe"]).to_frame("AP50 por classe"))
else:
    print("Sem split de teste (menos de 3 clipes com caixas): só a validação do treino foi usada.")

## 6. Análise do vídeo cirúrgico (clipe de teste)

Os frames do pacote **não são contínuos**: cada clipe tem poucas centenas de imagens espalhadas por mais de mil
posições, com lacunas irregulares. Por isso o vídeo de demonstração é apenas uma *montagem* (as lacunas somem) e
**não há eixo de tempo em segundos**. A coordenada temporal usada é o **número do frame original**; deslocamento
e ausência de instrumentos só são calculados entre frames **adjacentes** (diferença ≤ 1,5), e uma lacuna
interrompe a sequência. O detector roda no vídeo montado e os indicadores são calculados sobre as detecções.

> Os indicadores abaixo são **heurísticas de triagem**: o CholecT50 não tem rótulos de erro ou desvio
> cirúrgico, então só a detecção dos instrumentos (mAP, seção 5) pode ser validada contra o dataset.
> Frames sem anotação no JSON (sem rótulo) são excluídos da comparação com a anotação.


In [ ]:
from src.config import settings
from src.video.report import generate_video_report
from src.video.surgical_tools import SurgicalToolAnalyzer, analyze_detections

test_clips = stats["splits"].get("test", {}).get("videos") or stats["splits"]["val"]["videos"]
CLIP = test_clips[0]
clip = {s.name: s for s in discover_sources(RAW_DIR)}[CLIP]
frame_ids = sorted(clip.images)
DEMO_VIDEO = frames_to_video({i: clip.images[i] for i in frame_ids}, f"data/demo/{CLIP}.mp4", fps=1)
annotated = {fid for fid in frame_ids if fid in clip.annotations}      # frames COM rótulo no JSON
print(f"Clipe de teste: {CLIP} | {len(frame_ids)} frames | {len(annotated)} anotados | "
      f"{len(frame_ids) - len(annotated)} sem anotação (excluídos da comparação) | vídeo de demonstração: {DEMO_VIDEO}")

analyzer = SurgicalToolAnalyzer(conf=CONF)
frames = analyzer.detect(str(DEMO_VIDEO), timestamps=frame_ids)      # t = nº do frame original, não segundos
PID = patient_id_for(CLIP)
res = analyze_detections(frames, PID, max_idle_s=MAX_IDLE_S, time_unit="frame")
print(json.dumps(res.summary, indent=2, ensure_ascii=False))
display(findings_table(res.findings))

### 6.1 Linha do tempo dos instrumentos e comparação com a anotação

In [ ]:
t = [f.t for f in frames]
fig, ax = plt.subplots(2, 1, figsize=(13, 6), sharex=True, gridspec_kw={"height_ratios": [3, 1.6]})
for yi, cname in enumerate(NAMES):
    xs = [f.t for f in frames if any(d.cls == cname for d in f.dets)]
    ax[0].scatter(xs, [yi] * len(xs), s=16, color=np.array(PALETTE[yi % len(PALETTE)]) / 255)
ax[0].set_yticks(range(len(NAMES))); ax[0].set_yticklabels(NAMES)
for f in frames:
    if not f.visible:
        ax[0].axvspan(f.t - 0.5, f.t + 0.5, color="grey", alpha=0.25)
for fd in res.findings:
    if fd.kind == "ausencia_prolongada_de_instrumentos":
        ax[0].axvspan(fd.details["start_s"], fd.details["start_s"] + fd.details["duration_s"], color="red", alpha=0.12)
    if fd.kind == "movimento_brusco_instrumento":
        for ex in fd.details["examples"]:
            ax[0].axvline(ex["t"], color="orange", ls="--", lw=1)
ax[0].set_title("Instrumentos detectados (cinza = fora do corpo; vermelho = sem instrumentos; laranja = movimento brusco)")

# só frames COM anotação entram na comparação (frames ausentes do JSON não têm rótulo, não são "0 instrumentos")
ann_idx = [i for i, fid in enumerate(frame_ids) if fid in annotated]
t_ann = [t[i] for i in ann_idx]
pred_counts = [len(frames[i].dets) for i in ann_idx]
gt_counts = [len(clip.boxes(frame_ids[i])[0]) for i in ann_idx]
ax[1].scatter(t_ann, gt_counts, label="anotado (caixas)", s=12)
ax[1].scatter(t_ann, pred_counts, label="previsto", s=12, marker="x")
ax[1].set_ylabel("instrumentos"); ax[1].set_xlabel("número do frame original (frames não são contínuos)"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

vis = [k for k, i in enumerate(ann_idx) if frames[i].visible and clip.boxes(frame_ids[i])[1] != "ambiguous"]
mae = float(np.mean([abs(pred_counts[k] - gt_counts[k]) for k in vis])) if vis else float("nan")
print(f"Erro absoluto médio na contagem de instrumentos por frame: {mae:.2f} (frames comparáveis: {len(vis)} de {len(frame_ids)})")

### 6.2 Detecções sobre os frames (anotação em branco, previsão colorida)

In [ ]:
from ultralytics import YOLO
model = YOLO(settings.surgical_model)

positives = [fid for fid in frame_ids if fid in annotated and clip.boxes(fid)[1] == "positive"]
pick = positives[:: max(1, len(positives) // 6)][:6]
fig, axes = plt.subplots(2, 3, figsize=(15, 6))
for ax, fid in zip(axes.ravel(), pick):
    bgr = cv2.imread(str(clip.images[fid]))
    r = model(bgr, conf=CONF, verbose=False)[0]
    preds = [(int(b.cls), *b.xywhn[0].tolist()) for b in r.boxes]
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    rgb = draw_boxes(rgb, clip.boxes(fid)[0], NAMES, thickness=1, color=(255, 255, 255), prefix="GT ")
    rgb = draw_boxes(rgb, preds, NAMES, thickness=3)
    ax.imshow(rgb); ax.set_title(f"{CLIP} · frame {fid}", fontsize=9); ax.axis("off")
plt.suptitle("Branco fino = anotação (GT) · colorido grosso = previsão do YOLOv8"); plt.tight_layout(); plt.show()

### 6.3 Vídeo anotado

In [ ]:
if RENDER_VIDEO:
    raw_out = Path("data/demo") / f"{CLIP}_anotado_raw.mp4"
    h, w = cv2.imread(str(clip.images[frame_ids[0]])).shape[:2]
    vw = cv2.VideoWriter(str(raw_out), cv2.VideoWriter_fourcc(*"mp4v"), 2, (w, h))   # montagem a 2 fps (NÃO é tempo real: os frames são esparsos)
    for fid in frame_ids:
        bgr = cv2.imread(str(clip.images[fid]))
        r = model(bgr, conf=CONF, verbose=False)[0]
        preds = [(int(b.cls), *b.xywhn[0].tolist()) for b in r.boxes]
        rgb = draw_boxes(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB), preds, NAMES, thickness=3)
        vw.write(cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))
    vw.release()
    final = Path("data/demo") / f"{CLIP}_anotado.mp4"
    try:   # re-codifica em H.264 para tocar no navegador
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(raw_out), "-vcodec", "libx264",
                        "-pix_fmt", "yuv420p", str(final)], check=True)
    except Exception as e:  # noqa: BLE001
        print("ffmpeg indisponível; usando o arquivo sem re-codificar:", e)
        final = raw_out
    size_mb = final.stat().st_size / 1e6
    print(f"Vídeo anotado: {final} ({size_mb:.1f} MB)")
    if size_mb < 40:
        display(Video(str(final), embed=True, width=720))
    else:
        print("Arquivo grande demais para embutir; baixe-o pelo painel de arquivos.")

### 6.4 Relatório automático

In [ ]:
report_path = generate_video_report(patient_id_for(CLIP), str(DEMO_VIDEO), res.findings, summary=res.summary)
print("Relatório salvo em", report_path)
display(HTML(report_path.read_text(encoding="utf-8")))

### 6.5 Indicadores em todos os clipes (um ID de paciente por clipe)

Cada clipe com caixas anotadas é tratado como um paciente separado. Os indicadores são calculados com o ID do clipe; os achados não são misturados entre clipes.

In [ ]:
rows = []
for s in discover_sources(RAW_DIR):
    fids = sorted(s.images)
    ann = [fid for fid in fids if fid in s.annotations]
    vid = frames_to_video({i: s.images[i] for i in fids}, f"data/demo/{s.name}.mp4", fps=1)
    fr = analyzer.detect(str(vid), timestamps=fids)
    r = analyze_detections(fr, patient_id_for(s.name), max_idle_s=MAX_IDLE_S, time_unit="frame")
    rows.append({"paciente (ID)": patient_id_for(s.name), "frames": len(fids), "anotados": len(ann),
                 "fora_do_campo_%": r.summary.get("deteccoes_fora_do_campo_pct"),
                 "achados": ", ".join(sorted({f.kind for f in r.findings})) or "nenhum"})
display(pd.DataFrame(rows))
print("Cada linha é um clipe (ID de paciente) separado. Os achados são heurísticas de triagem, não diagnósticos.")


## 7. (Opcional) Fusão multimodal e alerta

Mostra o fluxo final do projeto: o achado de vídeo é fundido com sinais vitais e prescrições e gera o alerta
para a equipe médica. **Os sinais vitais e as prescrições são sintéticos.** Eles são gerados apenas para o ID do
clipe de teste, só para demonstrar o mecanismo: **não pertencem a esse paciente** e não são uma conclusão clínica.


In [ ]:
if RUN_FUSION:
    subprocess.run([sys.executable, "scripts/generate_synthetic_data.py", "--patient", patient_id_for(CLIP)], check=True)
    logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True)
    from src.pipeline import run
    PID = patient_id_for(CLIP)
    assessment = run(PID, video=str(DEMO_VIDEO), vitals=f"data/samples/vitals_{PID}.csv",
                     prescriptions=f"data/samples/prescriptions_{PID}.csv", dispatch=True, video_type="surgery",
                     video_timestamps=frame_ids)
    COLORS = {"baixa": "#2e7d32", "media": "#f9a825", "alta": "#ef6c00", "critica": "#c62828"}
    mods = ", ".join(sorted({f.modality.value for f in assessment.findings})) or "nenhuma"
    display(HTML(
        f"<div style='padding:12px 16px;border-radius:8px;background:{COLORS[assessment.severity.value]};"
        f"color:white;font-size:18px'>Paciente {assessment.patient_id} — risco <b>{assessment.severity.value.upper()}</b> "
        f"(score {assessment.score:.2f}) · modalidades: {mods}</div>"))
    display(findings_table(assessment.findings).head(12))
    if settings.alerts_path.exists():
        alerts = [json.loads(l) for l in settings.alerts_path.read_text(encoding="utf-8").splitlines() if l.strip()]
        display(pd.DataFrame(alerts)[["timestamp", "patient_id", "severity", "score", "modalities"]].tail(5))
else:
    print("Fusão desativada (RUN_FUSION=False).")

## 8. Salvar resultados no Google Drive

O disco do Colab é apagado ao fim da sessão. Guarde pesos, métricas, relatórios e o vídeo anotado.

In [ ]:
if USE_DRIVE and IN_COLAB:
    if not os.path.isdir("/content/drive/MyDrive"):
        from google.colab import drive
        drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)
    for d in ["models", "runs", "data/reports", "data/alerts", "data/demo"]:
        if os.path.isdir(d):
            shutil.copytree(d, os.path.join(DRIVE_DIR, os.path.basename(d)), dirs_exist_ok=True)
    if Path(YOLO_DIR, "dataset_stats.json").exists():
        shutil.copy2(Path(YOLO_DIR, "dataset_stats.json"), DRIVE_DIR)
    print("Resultados copiados para", DRIVE_DIR)
else:
    print("Nada copiado (USE_DRIVE=False ou fora do Colab).")